## dim_product

In [0]:
%sql

CREATE OR REPLACE TABLE adb_adventureworks_project.gold.dim_product
USING DELTA
AS
SELECT
    p.ProductKey,
    p.ProductSKU,
    p.ProductName,
    p.ModelName,
    p.ProductColor,
    p.ProductSize,
    p.ProductStyle,
    sc.ProductSubcategoryKey,
    sc.SubcategoryName,
    c.ProductCategoryKey,
    c.CategoryName,

    CAST(p.ProductPrice AS DECIMAL(18,4)) AS ProductPrice,
    CAST(p.ProductCost AS DECIMAL(18,4)) AS ProductCost
FROM 
    adb_adventureworks_project.silver.products_silver_table p
LEFT JOIN 
    adb_adventureworks_project.silver.product_subcategories_silver_table sc
ON 
    p.ProductSubcategoryKey = sc.ProductSubcategoryKey
LEFT JOIN 
    adb_adventureworks_project.silver.product_categories_silver_table c
ON 
    sc.ProductCategoryKey = c.ProductCategoryKey;


## dim_customer

In [0]:
%sql 

CREATE OR REPLACE TABLE adb_adventureworks_project.gold.dim_customer
USING DELTA
AS
SELECT
    CustomerKey,
    FullName,
    BirthDate,
    Gender,
    MaritalStatus,

    TRY_CAST(
        REGEXP_REPLACE(AnnualIncome, '[^0-9.]', '')
        AS DECIMAL(12,2)
    ) AS AnnualIncome,

    TotalChildren,
    EducationLevel,
    Occupation,
    HomeOwner

FROM 
    adb_adventureworks_project.silver.customers_silver_table;


## dim_territory

In [0]:
%sql

CREATE OR REPLACE TABLE adb_adventureworks_project.gold.dim_territory
USING DELTA
AS
SELECT
    SalesTerritoryKey AS TerritoryKey,
    Region,
    Country,
    Continent
FROM 
    adb_adventureworks_project.silver.territories_silver_table;


## dim_date

In [0]:
%sql

CREATE OR REPLACE TABLE adb_adventureworks_project.gold.dim_date
USING DELTA
AS
SELECT
    CAST(DATE_FORMAT(FullDate, 'yyyyMMdd') AS INT) AS DateKey,
    FullDate,

    YEAR(FullDate) AS Year,
    QUARTER(FullDate) AS Quarter,
    MONTH(FullDate) AS Month,
    DATE_FORMAT(FullDate, 'MMMM') AS MonthName,
    DATE_FORMAT(FullDate, 'yyyy-MM') AS YearMonth,

    DAY(FullDate) AS DayOfMonth,
    DAYOFWEEK(FullDate) AS DayOfWeek,
    DATE_FORMAT(FullDate, 'EEEE') AS DayName,

    CASE
        WHEN DAYOFWEEK(FullDate) IN (1, 7)
        THEN TRUE
        ELSE FALSE
    END AS IsWeekend

FROM (
    SELECT EXPLODE(
        SEQUENCE(
            DATE'2015-01-01',
            DATE'2018-12-31',
            INTERVAL 1 DAY
        )
    ) AS FullDate
);